In [28]:
# This notebook demonstrates common PySpark DataFrame operations and data processing.
from pyspark.sql import SparkSession
session_spark = SparkSession.builder.appName("DataProcessingExample").getOrCreate()

In [29]:
# Step 1: Load initial sales data from a CSV file
sales_df = session_spark.read.csv("source.txt", header=True, inferSchema=True)

In [30]:
# Display a preview of the loaded sales data
sales_df.show()

+-------+----------+-------------+-----------+------+----------+---------+------+------+--------+
|user_id|product_id|     old_name|   category| price|base_price|   status|amount|region|priority|
+-------+----------+-------------+-----------+------+----------+---------+------+------+--------+
|    101|      P001|       Laptop|Electronics|1200.5|    1200.5|Completed|  1500| North|    High|
|    102|      P002|   Smartphone|Electronics| 800.0|     800.0|  Pending|   900| South|  Medium|
|   NULL|      P003|   Headphones|Electronics| 150.0|     150.0|Completed|  1100| North|     Low|
|    104|      P004|      T-Shirt|    Apparel|  25.0|      25.0|Completed|    50|  East|     Low|
|    105|      P005| Coffee Maker|       Home| 99.99|     99.99|Cancelled|   120|  West|    High|
|    106|      P006|      Trimmer|Electronics|  45.5|      45.5|Completed|  2000|  West|    High|
|    107|      P007|  Smart Watch|Electronics| 250.0|     250.0|Completed|  1300| North|    High|
|   NULL|      P008|

In [31]:
# Count the total number of records in the sales DataFrame
sales_df.count()

8

In [32]:
# Display the schema of the sales DataFrame to understand data types
sales_df.printSchema()

root
 |-- user_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- old_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: double (nullable = true)
 |-- base_price: double (nullable = true)
 |-- status: string (nullable = true)
 |-- amount: integer (nullable = true)
 |-- region: string (nullable = true)
 |-- priority: string (nullable = true)



In [33]:
# Create a new DataFrame and save it as a Parquet file
product_data = [
    (1, "Laptop", 55000),
    (2, "Mouse", 800),
    (3, "Keyboard", 1500)
]
product_info_df = session_spark.createDataFrame(product_data, ["id", "product", "price"])
product_info_df.write.mode("overwrite").parquet("product_details.parquet")

In [34]:
# Read the Parquet file back into a DataFrame
product_parquet_df = session_spark.read.parquet("product_details.parquet")
product_parquet_df.show()

+---+--------+-----+
| id| product|price|
+---+--------+-----+
|  2|   Mouse|  800|
|  3|Keyboard| 1500|
|  1|  Laptop|55000|
+---+--------+-----+



In [35]:
# Display the schema of the DataFrame loaded from the Parquet file
product_parquet_df.printSchema()

root
 |-- id: long (nullable = true)
 |-- product: string (nullable = true)
 |-- price: long (nullable = true)



In [36]:
# Step 2: Filter and select data - focusing on 'Electronics' category
electronics_df = sales_df.select("user_id", "base_price","category").filter(sales_df.category == "Electronics")
electronics_df.show()

+-------+----------+-----------+
|user_id|base_price|   category|
+-------+----------+-----------+
|    101|    1200.5|Electronics|
|    102|     800.0|Electronics|
|   NULL|     150.0|Electronics|
|    106|      45.5|Electronics|
|    107|     250.0|Electronics|
+-------+----------+-----------+



In [37]:
# Filter data for 'Completed' status and amount greater than 1000
completed_high_amount_sales_df = sales_df.filter((sales_df.status == "Completed") & (sales_df.amount > 1000))
completed_high_amount_sales_df.show()

+-------+----------+-----------+-----------+------+----------+---------+------+------+--------+
|user_id|product_id|   old_name|   category| price|base_price|   status|amount|region|priority|
+-------+----------+-----------+-----------+------+----------+---------+------+------+--------+
|    101|      P001|     Laptop|Electronics|1200.5|    1200.5|Completed|  1500| North|    High|
|   NULL|      P003| Headphones|Electronics| 150.0|     150.0|Completed|  1100| North|     Low|
|    106|      P006|    Trimmer|Electronics|  45.5|      45.5|Completed|  2000|  West|    High|
|    107|      P007|Smart Watch|Electronics| 250.0|     250.0|Completed|  1300| North|    High|
+-------+----------+-----------+-----------+------+----------+---------+------+------+--------+



In [38]:
# Step 3: Data Transformations

# Rename the 'product_price' column to 'price' (assuming 'old_name' was meant to be 'product_price' in the original df)
# Note: The original 'df' already had 'price'. This line might cause an issue if 'product_price' doesn't exist.
# Assuming the intent was to operate on 'sales_df' and handle potential renaming if it came from a different source.
# For now, will apply to `sales_df` and assign to `processed_sales_df` to keep the flow.
processed_sales_df = sales_df.withColumnRenamed("product_price", "price")

In [39]:
processed_sales_df.show()

+-------+----------+-------------+-----------+------+----------+---------+------+------+--------+
|user_id|product_id|     old_name|   category| price|base_price|   status|amount|region|priority|
+-------+----------+-------------+-----------+------+----------+---------+------+------+--------+
|    101|      P001|       Laptop|Electronics|1200.5|    1200.5|Completed|  1500| North|    High|
|    102|      P002|   Smartphone|Electronics| 800.0|     800.0|  Pending|   900| South|  Medium|
|   NULL|      P003|   Headphones|Electronics| 150.0|     150.0|Completed|  1100| North|     Low|
|    104|      P004|      T-Shirt|    Apparel|  25.0|      25.0|Completed|    50|  East|     Low|
|    105|      P005| Coffee Maker|       Home| 99.99|     99.99|Cancelled|   120|  West|    High|
|    106|      P006|      Trimmer|Electronics|  45.5|      45.5|Completed|  2000|  West|    High|
|    107|      P007|  Smart Watch|Electronics| 250.0|     250.0|Completed|  1300| North|    High|
|   NULL|      P008|

In [40]:
processed_sales_df.printSchema()

root
 |-- user_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- old_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: double (nullable = true)
 |-- base_price: double (nullable = true)
 |-- status: string (nullable = true)
 |-- amount: integer (nullable = true)
 |-- region: string (nullable = true)
 |-- priority: string (nullable = true)



In [41]:
# Change the data type of the 'price' column to String
from pyspark.sql.functions import col
processed_sales_df = processed_sales_df.withColumn("price", col("price").cast("string"))

In [42]:
# Display the schema after casting 'price' to String
processed_sales_df.printSchema()

root
 |-- user_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- old_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: string (nullable = true)
 |-- base_price: double (nullable = true)
 |-- status: string (nullable = true)
 |-- amount: integer (nullable = true)
 |-- region: string (nullable = true)
 |-- priority: string (nullable = true)



In [43]:
# Convert the 'price' column back to Double type
from pyspark.sql.functions import col
processed_sales_df = processed_sales_df.withColumn("price", col("price").cast("double"))

In [44]:
# Display the schema after casting 'price' to Double
processed_sales_df.printSchema()

root
 |-- user_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- old_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: double (nullable = true)
 |-- base_price: double (nullable = true)
 |-- status: string (nullable = true)
 |-- amount: integer (nullable = true)
 |-- region: string (nullable = true)
 |-- priority: string (nullable = true)



In [45]:
# Create a new column 'final_price' by applying a calculation to 'base_price'
from pyspark.sql.functions import col
processed_sales_df = processed_sales_df.withColumn("final_price", col("base_price") * 1.18)
processed_sales_df.show()

+-------+----------+-------------+-----------+------+----------+---------+------+------+--------+------------------+
|user_id|product_id|     old_name|   category| price|base_price|   status|amount|region|priority|       final_price|
+-------+----------+-------------+-----------+------+----------+---------+------+------+--------+------------------+
|    101|      P001|       Laptop|Electronics|1200.5|    1200.5|Completed|  1500| North|    High|           1416.59|
|    102|      P002|   Smartphone|Electronics| 800.0|     800.0|  Pending|   900| South|  Medium|             944.0|
|   NULL|      P003|   Headphones|Electronics| 150.0|     150.0|Completed|  1100| North|     Low|             177.0|
|    104|      P004|      T-Shirt|    Apparel|  25.0|      25.0|Completed|    50|  East|     Low|              29.5|
|    105|      P005| Coffee Maker|       Home| 99.99|     99.99|Cancelled|   120|  West|    High|117.98819999999999|
|    106|      P006|      Trimmer|Electronics|  45.5|      45.5|

In [46]:
# Step 4: Handle missing values by dropping rows with any NULLs
cleaned_sales_df = processed_sales_df.dropna()
cleaned_sales_df.show()

+-------+----------+------------+-----------+------+----------+---------+------+------+--------+------------------+
|user_id|product_id|    old_name|   category| price|base_price|   status|amount|region|priority|       final_price|
+-------+----------+------------+-----------+------+----------+---------+------+------+--------+------------------+
|    101|      P001|      Laptop|Electronics|1200.5|    1200.5|Completed|  1500| North|    High|           1416.59|
|    102|      P002|  Smartphone|Electronics| 800.0|     800.0|  Pending|   900| South|  Medium|             944.0|
|    104|      P004|     T-Shirt|    Apparel|  25.0|      25.0|Completed|    50|  East|     Low|              29.5|
|    105|      P005|Coffee Maker|       Home| 99.99|     99.99|Cancelled|   120|  West|    High|117.98819999999999|
|    106|      P006|     Trimmer|Electronics|  45.5|      45.5|Completed|  2000|  West|    High|             53.69|
|    107|      P007| Smart Watch|Electronics| 250.0|     250.0|Completed

In [47]:
# Step 5: Perform wide transformation - group by 'category' and aggregate 'price'
from pyspark.sql.functions import sum
category_total_price_df = processed_sales_df.groupBy("category").agg(sum("price").alias("total_category_price"))
category_total_price_df.show()

+-----------+--------------------+
|   category|total_category_price|
+-----------+--------------------+
|       Home|               99.99|
|     Sports|                85.0|
|    Apparel|                25.0|
|Electronics|              2446.0|
+-----------+--------------------+



In [48]:
# Demonstrate Predicate Pushdown: Filter sales with 'price' greater than 1000
from pyspark.sql.functions import col
high_price_sales_df = processed_sales_df.filter(col("price") > 1000)
high_price_sales_df.show()

+-------+----------+--------+-----------+------+----------+---------+------+------+--------+-----------+
|user_id|product_id|old_name|   category| price|base_price|   status|amount|region|priority|final_price|
+-------+----------+--------+-----------+------+----------+---------+------+------+--------+-----------+
|    101|      P001|  Laptop|Electronics|1200.5|    1200.5|Completed|  1500| North|    High|    1416.59|
+-------+----------+--------+-----------+------+----------+---------+------+------+--------+-----------+



In [49]:
# Step 6: Build a data processing pipeline
from pyspark.sql.functions import col
electronics_pipeline_df = (
    session_spark.read.csv("source.txt", header=True, inferSchema=True)
    .withColumnRenamed("product_price", "price") # Assuming 'product_price' exists or needs renaming
    .withColumn("price", col("price").cast("double"))
    .withColumn("final_price", col("base_price") * 1.18)
    .filter(col("category") == "Electronics")
)
electronics_pipeline_df.dropna().show()

+-------+----------+-----------+-----------+------+----------+---------+------+------+--------+-----------+
|user_id|product_id|   old_name|   category| price|base_price|   status|amount|region|priority|final_price|
+-------+----------+-----------+-----------+------+----------+---------+------+------+--------+-----------+
|    101|      P001|     Laptop|Electronics|1200.5|    1200.5|Completed|  1500| North|    High|    1416.59|
|    102|      P002| Smartphone|Electronics| 800.0|     800.0|  Pending|   900| South|  Medium|      944.0|
|    106|      P006|    Trimmer|Electronics|  45.5|      45.5|Completed|  2000|  West|    High|      53.69|
|    107|      P007|Smart Watch|Electronics| 250.0|     250.0|Completed|  1300| North|    High|      295.0|
+-------+----------+-----------+-----------+------+----------+---------+------+------+--------+-----------+



In [50]:
# Step 7: Save processed data into CSV Format
electronics_pipeline_df.write.mode("overwrite").option("header", True).csv("processed_electronics_data_csv")

In [51]:
# Save processed data into Parquet Format
electronics_pipeline_df.write.mode("overwrite").parquet("processed_electronics_data_parquet")

In [52]:
# Read the newly saved Parquet file and display its contents
read_parquet_output_df = session_spark.read.parquet("processed_electronics_data_parquet")
read_parquet_output_df.show(5)

+-------+----------+-----------+-----------+------+----------+---------+------+------+--------+-----------+
|user_id|product_id|   old_name|   category| price|base_price|   status|amount|region|priority|final_price|
+-------+----------+-----------+-----------+------+----------+---------+------+------+--------+-----------+
|    101|      P001|     Laptop|Electronics|1200.5|    1200.5|Completed|  1500| North|    High|    1416.59|
|    102|      P002| Smartphone|Electronics| 800.0|     800.0|  Pending|   900| South|  Medium|      944.0|
|   NULL|      P003| Headphones|Electronics| 150.0|     150.0|Completed|  1100| North|     Low|      177.0|
|    106|      P006|    Trimmer|Electronics|  45.5|      45.5|Completed|  2000|  West|    High|      53.69|
|    107|      P007|Smart Watch|Electronics| 250.0|     250.0|Completed|  1300| North|    High|      295.0|
+-------+----------+-----------+-----------+------+----------+---------+------+------+--------+-----------+

